# per-rank-cuda-device — worked example 1: F-string builds unique device per rank

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `per-rank-cuda-device`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Each process in a distributed job gets assigned exactly one GPU, identified by `torch.device(f'cuda:{rank}')`. Using an f-string with the rank integer guarantees that rank 0 gets `cuda:0`, rank 1 gets `cuda:1`, and so on. This uniqueness invariant — no two ranks share a device — prevents GPU memory contention and ensures gradients accumulate on the correct device.

## Worked solution

**Step 1 — Build the device string.** `f'cuda:{rank}'` is a simple f-string substitution. For rank 2, this produces the string `'cuda:2'`. Wrapping it in `torch.device(...)` creates the device object that PyTorch's `.to()` and tensor constructors accept.

**Step 2 — Verify uniqueness.** For a 4-rank job, the four devices are `cuda:0`, `cuda:1`, `cuda:2`, `cuda:3`. We confirm this by building all four and checking that the `.index` attributes are all distinct.

**Step 3 — is_master flag.** Rank 0 is the 'master' rank responsible for checkpointing and logging. `is_master = (rank == 0)` is a simple boolean that downstream code uses as a guard.

**Step 4 — device_str for logging.** Storing the string `f'cuda:{rank}'` alongside the device object makes log messages human-readable without calling `str(device)` each time.

In [ ]:
import torch as t

def build_rank_context(rank: int, world_size: int) -> dict:
    """Build per-rank context dict with unique CUDA device."""
    device = t.device(f'cuda:{rank}')   # unique per rank via f-string
    return {
        'rank': rank,
        'world_size': world_size,
        'device': device,
        'is_master': rank == 0,
        'device_str': f'cuda:{rank}',
    }

# Build contexts for all 4 ranks and verify uniqueness
world_size = 4
contexts = [build_rank_context(r, world_size) for r in range(world_size)]

device_indices = {ctx['device'].index for ctx in contexts}
print(f'Device indices: {sorted(device_indices)}')    # {0, 1, 2, 3}
print(f'All unique: {len(device_indices) == world_size}')  # True
print(f'Rank 0 is master: {contexts[0]["is_master"]}')     # True
print(f'Rank 2 is master: {contexts[2]["is_master"]}')     # False
print(f'Rank 1 device: {contexts[1]["device_str"]}')       # cuda:1